# Structured Data — Exercise

**Task:** use message prefilling and stop sequences *only* to get **three different AWS CLI commands** in a single response — no comments, no explanation.

**Hint:** prefilling isn't limited to ` ``` `. You can prefill *any* text to steer how Claude starts.

**The trick:**
- Prefill the assistant turn with `"1."` → Claude thinks it already started a numbered list and jumps straight to the first command (no "Here are three commands:" preamble).
- Set `stop_sequences=["4."]` → the moment Claude starts a fourth item, generation stops — so you get exactly three and nothing after them.

> Runs on `claude-haiku-4-5-20251001` because it uses assistant prefilling (flagship models 400 on prefill).

## Setup

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5-20251001"   # supports assistant prefilling


def get_text(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""


def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return get_text(message)


print(f"Ready. Model: {model}")

Ready. Model: claude-haiku-4-5-20251001


## Solution

The prefilled `"1."` isn't returned in the response (the API sends only the continuation), so we prepend it back to reconstruct the full list.

In [2]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "1.")          # prefill: force a numbered list, no preamble

text = chat(messages, stop_sequences=["4."])    # stop before a 4th item / any trailing text

# Prepend the prefill back to see all three commands
result = ("1." + text).strip()
print(result)

1.
```bash
aws s3 ls s3://my-bucket
```

2.
```bash
aws ec2 describe-instances --region us-east-1
```

3.
```bash
aws dynamodb list-tables
```


## Why it works

- **`add_assistant_message(messages, "1.")`** — Claude sees a response that has already begun with `1.`, so it can't add a "Here are three commands:" preamble. It just continues the list.
- **`stop_sequences=["4."]`** — three commands numbered `1.`, `2.`, `3.` naturally lead Claude toward `4.`; the stop sequence fires there, chopping off the fourth item and any explanation Claude might have added afterward.
- **`"1." + text`** — the model's response starts at command 1's *text* (the prefill itself isn't echoed back), so we glue the `1.` back on for a complete, clean list.